# Эмпирическое исследование сортировки строк

Посылки CodeForces:

- A1m / merge: `375960976`
- A1q / quick: `375962191`
- A1r / radix: `375963006`
- A1rq / qradix: `375964008`

Ссылка на публичный репозиторий: `TODO: добавить после публикации на GitHub`


## Методика

`Benchmark` использует строки длиной от 10 до 200 символов над алфавитом из 74 символов: `A..Z`, `a..z`, `0..9` и `!@#%:;^&*()-`. Для каждого типа данных генерируется один массив из 3000 строк, после чего тестируются его префиксы размеров 100, 200, ..., 3000. Каждый алгоритм запускается 30 раз на свежей копии одних и тех же данных.

Алгоритмы:

- standard quicksort;
- standard mergesort;
- ternary string quicksort;
- LCP-aware string mergesort;
- MSD radix sort;
- MSD radix sort с переключением на string quicksort для фрагментов меньше 74 элементов.


In [1]:
%use dataframe
%use kandy

import java.io.File

data class SummaryRow(
    val dataset: String,
    val algorithm: String,
    val size: Int,
    val avgTimeUs: Double,
    val avgCharComparisons: Double,
    val avgCharInspections: Double,
    val avgRadixCalls: Double,
    val avgQuickSwitches: Double,
)

fun loadSummary(path: String): List<SummaryRow> {
    val lines = File(path).readLines().filter { it.isNotBlank() }
    return lines.drop(1).map { line ->
        val parts = line.split(',')
        SummaryRow(
            dataset = parts[0],
            algorithm = parts[1],
            size = parts[2].toInt(),
            avgTimeUs = parts[3].toDouble(),
            avgCharComparisons = parts[4].toDouble(),
            avgCharInspections = parts[5].toDouble(),
            avgRadixCalls = parts[6].toDouble(),
            avgQuickSwitches = parts[7].toDouble(),
        )
    }
}

val summary = loadSummary("../results/summary.csv")
summary.take(10)

[SummaryRow(dataset=nearly_sorted, algorithm=lcp_mergesort, size=100, avgTimeUs=40.867, avgCharComparisons=1069.0, avgCharInspections=0.0, avgRadixCalls=0.0, avgQuickSwitches=0.0), SummaryRow(dataset=nearly_sorted, algorithm=lcp_mergesort, size=200, avgTimeUs=110.267, avgCharComparisons=2631.0, avgCharInspections=0.0, avgRadixCalls=0.0, avgQuickSwitches=0.0), SummaryRow(dataset=nearly_sorted, algorithm=lcp_mergesort, size=300, avgTimeUs=170.567, avgCharComparisons=4343.0, avgCharInspections=0.0, avgRadixCalls=0.0, avgQuickSwitches=0.0), SummaryRow(dataset=nearly_sorted, algorithm=lcp_mergesort, size=400, avgTimeUs=244.067, avgCharComparisons=6206.0, avgCharInspections=0.0, avgRadixCalls=0.0, avgQuickSwitches=0.0), SummaryRow(dataset=nearly_sorted, algorithm=lcp_mergesort, size=500, avgTimeUs=331.933, avgCharComparisons=8403.0, avgCharInspections=0.0, avgRadixCalls=0.0, avgQuickSwitches=0.0), SummaryRow(dataset=nearly_sorted, algorithm=lcp_mergesort, size=600, avgTimeUs=397.2, avgCharCo

In [2]:
val datasets = summary.map { it.dataset }.distinct()
val algorithms = summary.map { it.algorithm }.distinct()

println("Наборы данных: ${datasets.joinToString()}")
println("Алгоритмы: ${algorithms.joinToString()}")

Наборы данных: nearly_sorted, prefix_150, prefix_25, prefix_75, random, reverse
Алгоритмы: lcp_mergesort, msd_radix_sort, quick_msd_radix_sort, standard_mergesort, standard_quicksort, string_quicksort


In [3]:
fun metricValue(row: SummaryRow, metric: String): Double = when (metric) {
    "avgTimeUs" -> row.avgTimeUs
    "avgCharComparisons" -> row.avgCharComparisons
    "avgCharInspections" -> row.avgCharInspections
    "combined_char_metric" -> row.avgCharComparisons + row.avgCharInspections
    else -> error("Неизвестная метрика: $metric")
}

fun plotMetric(dataset: String, metric: String, titleText: String) {
    val rows = summary.filter { it.dataset == dataset }
    val frame = dataFrameOf(
        "size" to rows.map { it.size },
        "algorithm" to rows.map { it.algorithm },
        "value" to rows.map { metricValue(it, metric) },
    )

    DISPLAY(frame.groupBy("algorithm").plot {
        line {
            x("size")
            y("value")
            color("algorithm")
        }
        points {
            x("size")
            y("value")
            color("algorithm")
            size = 1.5
        }
        layout {
            title = titleText
            size = 1000 to 520
        }
    })
}

## Среднее время работы

Следующие графики сравнивают среднее время работы в микросекундах.


In [4]:
datasets.forEach { dataset ->
    plotMetric(dataset, "avgTimeUs", "Среднее время работы: $dataset")
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="eSELAF"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("eSELAF");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Среднее время работы: nearly_sorted"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_qu

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ci5WSO"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("ci5WSO");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Среднее время работы: prefix_150"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quick

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="aUcEQM"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("aUcEQM");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Среднее время работы: prefix_25"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicks

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="SS94Lk"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("SS94Lk");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Среднее время работы: prefix_75"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicks

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="baCibo"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("baCibo");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Среднее время работы: random"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicksort

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Bsx6AG"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("Bsx6AG");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Среднее время работы: reverse"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicksor

## Посимвольные сравнения и обращения

Алгоритмы на основе сравнений (`comparison-based algorithms`) учитывают прямые посимвольные сравнения. `Radix-based algorithms` не сравнивают строки тем же способом, поэтому для них учитываются обращения к символам. Общая метрика ниже удобна для отображения обеих групп алгоритмов на одном графике.


In [5]:
datasets.forEach { dataset ->
    plotMetric(dataset, "combined_char_metric", "Посимвольная метрика: $dataset")
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="9fUnTs"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("9fUnTs");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Посимвольная метрика: nearly_sorted"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_qu

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="MUoT0b"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("MUoT0b");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Посимвольная метрика: prefix_150"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quick

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JpHrl0"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("JpHrl0");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Посимвольная метрика: prefix_25"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicks

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="P6ui1J"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("P6ui1J");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Посимвольная метрика: prefix_75"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicks

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="crpWUE"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("crpWUE");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Посимвольная метрика: random"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicksort

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="rOI5yi"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1000.0, 
 height: 520.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("rOI5yi");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Посимвольная метрика: reverse"
},
"mapping":{
},
"data":{
"&merged_groups":["lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","lcp_mergesort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","quick_msd_radix_sort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_mergesort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","standard_quicksort","string_quicksort","string_quicksor

## Заметки для итогового анализа

- `Standard comparison sorts` имеют базовую оценку сложности `Omega(n log n)`, но при сортировке строк стоимость сравнения зависит еще и от длины общего префикса.
- `Ternary string quicksort` и `MSD radix sort` не сравнивают заново уже обработанные префиксы.
- На `prefix-heavy datasets` различие между обычными сортировками на основе сравнений и специализированными `string sorting algorithms` заметнее.
- Гибридный `MSD radix sort` уменьшает накладные расходы на маленьких рекурсивных фрагментах за счет переключения на `string quicksort` при размере фрагмента меньше мощности алфавита.
